<a href="https://colab.research.google.com/github/PeesapatiRohithSharma/Summer_2026/blob/main/ada_boost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [149]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
import pandas as pd

In [150]:
from google.colab import files
data = files.upload()

Saving processed.cleveland.data to processed.cleveland (4).data


In [151]:
df = pd.read_csv("processed.cleveland (4).data")
df.columns = [ "age","sex","cp","trestbps","chol","fbs","restecg",
 "thalach","exang","oldpeak","slope","ca","thal","target"]

In [152]:
df

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
1,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
2,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
3,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0
4,56.0,1.0,2.0,120.0,236.0,0.0,0.0,178.0,0.0,0.8,1.0,0.0,3.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
297,45.0,1.0,1.0,110.0,264.0,0.0,0.0,132.0,0.0,1.2,2.0,0.0,7.0,1
298,68.0,1.0,4.0,144.0,193.0,1.0,0.0,141.0,0.0,3.4,2.0,2.0,7.0,2
299,57.0,1.0,4.0,130.0,131.0,0.0,0.0,115.0,1.0,1.2,2.0,1.0,7.0,3
300,57.0,0.0,2.0,130.0,236.0,0.0,2.0,174.0,0.0,0.0,2.0,1.0,3.0,1


In [153]:
y = (df["target"] > 0).astype(int)

In [154]:
class AdaBoost:
  def __init__(self, n_trees = 100, max_depth = 4):
    self.n_trees = n_trees
    self.max_depth = max_depth
    self.trees = []
    self.alphas = []

  def fit(self, X, y):
    weights = np.full(len(y), 1/len(y))
    y_req = np.where(y == 0, -1, 1)
    for _ in range(self.n_trees):
      tree = DecisionTreeClassifier(max_depth=self.max_depth)
      tree.fit(X,y_req,sample_weight=weights)
      pred = tree.predict(X)
      incorrect = pred != y_req
      error = np.sum(weights*incorrect)
      error = max(error, 1e-10)
      alpha = 0.5 * (np.log((1-error)/error))
      self.trees.append(tree)
      self.alphas.append(alpha)
      weights = weights * (np.exp(-1 * alpha * pred * y_req))
      weights = weights/np.sum(weights)

  def predict(self, X):
    score = np.zeros(X.shape[0])
    for tree,alpha in zip(self.trees, self.alphas):
      score += tree.predict(X) * alpha
    preds = np.sign(score)
    return np.where(preds==-1, 0, 1)


In [155]:
model = AdaBoost()

In [156]:
df = df.replace("?", np.nan)

In [157]:
X = df.drop("target", axis=1).values

In [158]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [159]:
model.fit(X_train, y_train)

In [160]:
model.predict(X_test)

array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0,
       1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1,
       0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1])

In [161]:
accuracy = np.sum(model.predict(X_test) == y_test) / len(y_test)

In [162]:
accuracy * 100

np.float64(90.1639344262295)